# Chapter 8 — Containment Is Not Truth

**Book alignment:** Hallucination From First Principles, Chapter 8

**Question this notebook isolates:** Does a full-rank basis assign zero energy to contradictions alike, and does an inversion pair alias under the scalar while coordinates differ (reduction-pipeline loss)?

Synthetic embeddings below demonstrate the geometry type only and do not reproduce the book's empirical runs.

In [ ]:
import numpy as np

SEED = 11
rng = np.random.default_rng(SEED)
print("numpy", np.__version__, "seed", SEED)

## 1 — Full-rank limit: contradictions score zero like truths

With `rank(E) = d` and `r = d`, `B B^T = I`, so every unit claim has `H = 0`. A subspace that contains everything discriminates nothing.

In [ ]:
def hallucination_energy_full(claim_vec, evidence_vecs, rank_r):
    c = np.asarray(claim_vec, dtype=np.float64)
    E = np.asarray(evidence_vecs, dtype=np.float64)
    c = c / max(np.linalg.norm(c), 1e-12)
    E = E / np.where(np.linalg.norm(E, axis=1, keepdims=True) < 1e-12, 1.0,
                     np.linalg.norm(E, axis=1, keepdims=True))
    _, _, Vt = np.linalg.svd(E, full_matrices=False)
    basis = Vt[:min(rank_r, Vt.shape[0])].T
    coords = basis.T @ c
    return float(np.clip(1.0 - float(coords @ coords), 0.0, 1.0)), coords


d = 16
E_full = np.eye(d)
c_true = rng.standard_normal(d)
c_true = c_true / np.linalg.norm(c_true)
c_contra = -c_true  # opposite proposition, same span
c_fab = rng.standard_normal(d)
c_fab = c_fab / np.linalg.norm(c_fab)
h_true, _ = hallucination_energy_full(c_true, E_full, d)
h_contra, _ = hallucination_energy_full(c_contra, E_full, d)
h_fab, _ = hallucination_energy_full(c_fab, E_full, d)
print(f"H(true)={h_true:.6f} H(contradiction)={h_contra:.6f} H(fabricated)={h_fab:.6f}")

In [ ]:
assert h_true < 1e-9 and h_contra < 1e-9 and h_fab < 1e-9
print("Full-rank basis: true, contradictory, and fabricated claims all score zero.")

## 2 — Coordinate collapse: the scalar keeps how much, not where

Coordinates `z = B^T c` record where in the subspace the claim sits; energy keeps only `||z||^2`. Two orthogonal coordinate patterns can share one scalar.

In [ ]:
D2 = 8
E2 = np.zeros((2, D2))
E2[0, 0] = E2[1, 1] = 1.0
c_a = np.array([1.0, 0.0, 1.0, 0, 0, 0, 0, 0])
c_b = np.array([0.0, 1.0, 1.0, 0, 0, 0, 0, 0])
h_a, z_a = hallucination_energy_full(c_a, E2, 2)
h_b, z_b = hallucination_energy_full(c_b, E2, 2)
print(f"z_a={np.round(z_a, 4)} H={h_a:.4f}")
print(f"z_b={np.round(z_b, 4)} H={h_b:.4f}")
print(f"coord distance={float(np.linalg.norm(z_a - z_b)):.4f} scalar gap={abs(h_a - h_b):.6f}")

In [ ]:
assert abs(h_a - h_b) < 1e-9 and abs(h_a - 0.5) < 1e-9
assert float(np.linalg.norm(z_a - z_b)) > 0.5  # coordinates differ, scalar aliases
print("Aliasing: orthogonal placements in the subspace share H=0.5.")

## 3 — Inversion alias: opposite claims, same verdict

Since `(v·c)^2 = (v·(-c))^2`, a claim and its negation alias exactly under the scalar while pointing oppositely. No threshold on `H` can separate them: decision aliasing from missing information, not noise.

In [ ]:
TAU = 0.20
c_pos = np.array([0.6, 0.5, 0.8, 0, 0, 0, 0, 0])
c_neg = -c_pos  # polarity-reversed proposition
h_pos, z_pos = hallucination_energy_full(c_pos, E2, 2)
h_neg, z_neg = hallucination_energy_full(c_neg, E2, 2)
d_pos = "ACCEPT" if h_pos <= TAU else "REJECT"
d_neg = "ACCEPT" if h_neg <= TAU else "REJECT"
print(f"H(pos)={h_pos:.4f} H(neg)={h_neg:.4f} decisions={d_pos}/{d_neg}")
print(f"claim distance={float(np.linalg.norm(c_pos / np.linalg.norm(c_pos) - c_neg / np.linalg.norm(c_neg))):.4f}")

In [ ]:
assert abs(h_pos - h_neg) < 1e-9
assert (h_pos <= TAU) == (h_neg <= TAU)  # same decision, opposite claims
print("Inversion pair aliases: calibration moves the boundary, not the missing dimension.")

## What we earned

Containment is a permissive geometric relaxation, not a truth criterion: full rank contains contradictions, coordinates know where the scalar only knows how much, and inversion aliases prove a threshold cannot manufacture a discarded distinction.

Chapter 9 keeps containment as one typed sensor among many and adds the next axes: consistency (do stable relations hold?) and sensitivity (do decisive changes move the output?).